In [1]:
import os
for v in ("OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"

In [ ]:
# IMPORTS

import os
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.linear_model import HuberRegressor, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_fscore_support, average_precision_score
)

import xgboost as xgb
from xgboost import XGBRegressor, XGBClassifier
from IPython.display import display

In [ ]:
# CONFIG

DATA_PATH = "processed/chennai_aqi_era5_hourly.parquet"
OUT_DIR = "Timing_threads"
CTRL_CACHE_DIR = "huber_processed_leakfree"
warnings.filterwarnings("ignore")
SPLIT_TIME  = pd.Timestamp("2025-01-01 00:00:00")
L           = 24
H_LIST      = [1, 3, 6, 12, 24]
TAU_LIST    = [0, 1, 2, 3, 4, 6]
VALID_FRAC  = 0.20
GRAPH_K     = 4
RUN_TUNED   = True
JSO_POP     = 12
JSO_ITERS   = 15
RANDOM_SEED = 1
GATE_MODE = "fixed"
bounds = np.array([
    [1.0, 8.0],
    [0.01, 0.15],
    [-6.0, -2.0],
    [3.0, 10.0],
    [0.01, 0.20],
    [0.60, 1.00],
    [0.60, 1.00],
    [0.00, 10.00],
    [100.0, 800.0],
    [1.0, 5.0],
    [1.1, 10.0],
    [2.0, 30.0],
], dtype=float)

CTRL_CKPT_DIR   = os.path.join(OUT_DIR, "controls_ckpt")
CTRL_H_LIST     = list(H_LIST)
CTRL_TAU_STATIC = {}
CTRL_TAU_CSV    = ""
CTRL_PIECES     = ["xgb", "flat", "static"]
CTRL_VERIFY_FASTPATH = True
CTRL_FORCE      = False
CTRL_MERGE_DIRS = []
CTRL_AUTOMERGE  = True
CTRL_TAU_DYN   = {1: 6, 3: 4, 6: 2, 12: 6, 24: 2}
CTRL_JSO_SEEDS = [2, 3, 4]
CTRL_K_LIST    = [2, 3, 6, 8]
CTRL_CKPT_DIR = os.path.join(OUT_DIR, "controls_ckpt")
os.makedirs(OUT_DIR, exist_ok=True)

print("CONFIG")
print(f"  DATA_PATH      {DATA_PATH}")
print(f"  OUT_DIR        {OUT_DIR}")
print(f"  CTRL_CACHE_DIR {CTRL_CACHE_DIR}")
print(f"  CTRL_CKPT_DIR  {CTRL_CKPT_DIR}")
print(f"  CTRL_H_LIST    {CTRL_H_LIST}")
print(f"  GATE_MODE      {GATE_MODE!r}")
print(f"  JSO            pop={JSO_POP} iters={JSO_ITERS}: "
      f"{JSO_POP * (JSO_ITERS + 1)} evals per control per H")
for _p, _lab in ((DATA_PATH, "DATA_PATH"), (CTRL_CACHE_DIR, "CTRL_CACHE_DIR")):
    print(f"  {'OK  ' if os.path.exists(_p) else 'MISSING'} {_lab}")

CONFIG
  DATA_PATH      processed/chennai_aqi_era5_hourly.parquet
  OUT_DIR        Timing_threads
  CTRL_CACHE_DIR huber_processed_leakfree
  CTRL_CKPT_DIR  Timing_threads\controls_ckpt
  CTRL_H_LIST    [1, 3, 6, 12, 24]
  GATE_MODE      'fixed'
  JSO            pop=12 iters=15: 192 evals per control per H
  OK   DATA_PATH
  OK   CTRL_CACHE_DIR


In [ ]:
# HELPERS
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * R * np.arcsin(np.sqrt(a))

def bearing_radians(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return np.arctan2(y, x)

def flatten_window_per_node(X):
    S, Lx, N, F = X.shape
    return X.transpose(0, 2, 1, 3).reshape(S * N, Lx * F)

def compute_tail_metrics(y_true, y_pred, percentiles=(90, 95, 99)):
    rows = []
    for p in percentiles:
        thr = np.percentile(y_true, p)
        idx = y_true >= thr
        if idx.sum() == 0:
            rows.append((p, thr, np.nan, np.nan, 0))
        else:
            rows.append((
                p,
                thr,
                mean_absolute_error(y_true[idx], y_pred[idx]),
                np.sqrt(mean_squared_error(y_true[idx], y_pred[idx])),
                int(idx.sum())
            ))
    return rows

def compute_mfb_nmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mfb = np.mean(2.0 * (y_pred - y_true) / (y_pred + y_true + 1e-8))
    nmse = np.sum((y_pred - y_true) ** 2) / (np.sum(y_pred * y_true) + 1e-8)
    return mfb, nmse

def build_fill_values_from_train_timeline(X_train_timeline):
    fill_values = np.nanmedian(X_train_timeline, axis=0)
    global_fill = np.nanmedian(X_train_timeline.reshape(-1, X_train_timeline.shape[-1]), axis=0)
    global_fill = np.where(np.isnan(global_fill), 0.0, global_fill)
    for n in range(fill_values.shape[0]):
        for f in range(fill_values.shape[1]):
            if np.isnan(fill_values[n, f]):
                fill_values[n, f] = global_fill[f]
    fill_values = np.where(np.isnan(fill_values), 0.0, fill_values).astype(np.float32)
    return fill_values

def impute_windows(X, fill_values):
    X_imp = X.copy().astype(np.float32)
    S, Lx, N, F = X_imp.shape
    for s in range(S):
        for n in range(N):
            for f in range(F):
                series = pd.Series(X_imp[s, :, n, f], dtype="float32")
                series = series.ffill().bfill()
                arr = series.to_numpy(dtype=np.float32)
                if np.isnan(arr).any():
                    arr = np.where(np.isnan(arr), fill_values[n, f], arr)
                X_imp[s, :, n, f] = arr
    return X_imp

def build_nodes_edges(df, station_ids, k=4):
    coords = (
        df[["station_id", "lat", "lon"]]
        .drop_duplicates()
        .assign(station_id=lambda d: d["station_id"].astype(str))
    )
    nodes = coords[coords["station_id"].isin(station_ids)].copy()
    nodes = nodes.set_index("station_id").loc[station_ids].reset_index()
    nodes["node_id"] = np.arange(len(nodes))

    coords_arr = nodes[["lat", "lon"]].to_numpy(dtype=float)
    N = len(nodes)

    D = np.zeros((N, N), dtype=float)
    for i in range(N):
        D[i, :] = haversine_km_vec(coords_arr[i, 0], coords_arr[i, 1], coords_arr[:, 0], coords_arr[:, 1])

    sigma = np.median(D[D > 0])
    edges = []
    kk = min(k, N - 1)
    for i in range(N):
        nn = np.argsort(D[i])[1:kk + 1]
        for j in nn:
            w = np.exp(-(D[i, j] ** 2) / (2.0 * sigma ** 2))
            edges.append((i, j, float(w), float(D[i, j])))

    edges_df = pd.DataFrame(edges, columns=["src", "dst", "w_dist", "dist_km"])
    return nodes, edges_df

def build_static_adj_from_nodes(nodes_df, k=4, sigma_km=None, self_loops=False, row_normalize=True):
    coords = nodes_df[["lat", "lon"]].to_numpy(dtype=float)
    N = coords.shape[0]
    D = np.zeros((N, N), dtype=float)
    for i in range(N):
        D[i, :] = haversine_km_vec(coords[i, 0], coords[i, 1], coords[:, 0], coords[:, 1])

    if sigma_km is None:
        sigma_km = np.median(D[D > 0])

    A = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        nn = np.argsort(D[i])[1:min(k + 1, N)]
        for j in nn:
            A[i, j] = np.float32(np.exp(-(D[i, j] ** 2) / (2.0 * sigma_km ** 2)))

    if self_loops:
        np.fill_diagonal(A, 1.0)

    if row_normalize:
        rs = A.sum(axis=1, keepdims=True)
        A = np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)

    return A

def wind_bearing_toward(u, v):
    return np.arctan2(u, v)


def build_dynamic_adj(u_t, v_t, ctx, alpha=4.0, eps=0.05, gate_mode=None):
    if gate_mode is None:
        gate_mode = globals().get("GATE_MODE", "fixed")
    src, dst = ctx["src"], ctx["dst"]

    if gate_mode == "legacy":
        theta = np.arctan2(v_t[src], u_t[src])
        bearing = ctx["edge_bearing"]
    elif gate_mode in ("fixed", "shuffled"):
        theta = wind_bearing_toward(u_t[dst], v_t[dst])
        key = "edge_bearing_shuffled" if gate_mode == "shuffled" else "edge_bearing"
        if key not in ctx:
            raise KeyError(f"missing ctx key {key!r}")
        bearing = ctx[key] + np.pi
    else:
        raise ValueError(f"unknown gate_mode {gate_mode!r}")

    align = np.cos(theta - bearing)
    gate = 1.0 / (1.0 + np.exp(-alpha * align))
    gate = eps + (1.0 - eps) * gate
    w_dyn = ctx["w_dist"] * gate

    A = np.zeros((ctx["N"], ctx["N"]), dtype=np.float32)
    A[src, dst] = w_dyn.astype(np.float32)
    rs = A.sum(axis=1, keepdims=True)
    A = np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)
    return A
def make_graph_features_dynamic(X, ctx, tau=0, alpha=4.0, eps=0.05, gate_mode=None):
    S, Lx, N, F = X.shape
    Z = np.zeros((S, N, 2 * F), dtype=np.float32)
    tau = int(tau)
    for s in range(S):
        x_now = X[s, -1]
        x_lag = X[s, -1 - tau] if tau > 0 else x_now
        A = build_dynamic_adj(
            x_now[:, ctx["u_idx"]],
            x_now[:, ctx["v_idx"]],
            ctx,
            alpha=alpha,
            eps=eps,
            gate_mode=gate_mode
        )
        agg = A @ x_lag
        Z[s] = np.concatenate([x_now, agg], axis=-1)
    return Z
def make_graph_features_static(X, A_static, tau=0):
    S, Lx, N, F = X.shape
    Z = np.zeros((S, N, 2 * F), dtype=np.float32)
    tau = int(tau)
    for s in range(S):
        x_now = X[s, -1]
        x_lag = X[s, -1 - tau] if tau > 0 else x_now
        agg = A_static @ x_lag
        Z[s] = np.concatenate([x_now, agg], axis=-1)
    return Z

def summarize_regression(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred))
    }

def train_val_split_time_order(X_train_imp, Y_train, M_train, target_times_train, frac=0.2):
    n = X_train_imp.shape[0]
    val_size = max(1, int(np.ceil(frac * n)))
    idx_train = np.arange(0, n - val_size)
    idx_val   = np.arange(n - val_size, n)
    train_pack = {
        "X": X_train_imp[idx_train],
        "Y": Y_train[idx_train],
        "M": M_train[idx_train],
        "times": target_times_train[idx_train]
    }
    val_pack = {
        "X": X_train_imp[idx_val],
        "Y": Y_train[idx_val],
        "M": M_train[idx_val],
        "times": target_times_train[idx_val]
    }
    return train_pack, val_pack

def fit_flat_regressor(model, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test):
    Xtr = flatten_window_per_node(X_train_imp)
    Xte = flatten_window_per_node(X_test_imp)
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    model.fit(Xtr[mtr], ytr[mtr])
    pred_all = model.predict(Xte)
    return {
        "y_true": yte[mte],
        "y_pred": pred_all[mte],
        "model": model
    }

def fit_static_graph_regressor(model, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test, A_static, tau):
    Ztr = make_graph_features_static(X_train_imp, A_static, tau=tau)
    Zte = make_graph_features_static(X_test_imp, A_static, tau=tau)

    Xtr = Ztr.reshape(-1, Ztr.shape[-1])
    Xte = Zte.reshape(-1, Zte.shape[-1])
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    model.fit(Xtr[mtr], ytr[mtr])
    pred_all = model.predict(Xte)
    return {
        "y_true": yte[mte],
        "y_pred": pred_all[mte],
        "model": model
    }

def fit_huber_graph(X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test, ctx, tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4, gate_mode=None):
    Ztr = make_graph_features_dynamic(X_train_imp, ctx, tau=tau, alpha=alpha, eps=eps, gate_mode=gate_mode)
    Zte = make_graph_features_dynamic(X_test_imp, ctx, tau=tau, alpha=alpha, eps=eps, gate_mode=gate_mode)

    Xtr = Ztr.reshape(-1, Ztr.shape[-1])
    Xte = Zte.reshape(-1, Zte.shape[-1])
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    huber = Pipeline([
        ("scaler", StandardScaler()),
        ("huber", HuberRegressor(epsilon=1.35, alpha=huber_alpha, max_iter=500))
    ])
    huber.fit(Xtr[mtr], ytr[mtr])

    pred_train_all = huber.predict(Xtr)
    pred_test_all  = huber.predict(Xte)

    return {
        "y_true": yte[mte],
        "y_pred": pred_test_all[mte],
        "pred_train_all": pred_train_all,
        "pred_test_all": pred_test_all,
        "y_train_all": ytr,
        "y_test_all": yte,
        "mask_train": mtr,
        "mask_test": mte,
        "model": huber
    }

def fit_huber_hybrid(
    X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
    ctx, tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4,
    xgb_params=None, num_boost_round=400, seed=42,
    w_mid=2.0, w_danger=5.0, w_tail=10.0
):
    base = fit_huber_graph(
        X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
        ctx=ctx, tau=tau, alpha=alpha, eps=eps, huber_alpha=huber_alpha
    )

    ytr_all = base["y_train_all"]
    pred_tr_all = base["pred_train_all"]
    mtr = base["mask_train"]
    mte = base["mask_test"]

    res_train = ytr_all - pred_tr_all
    Xgb_train = np.asarray(flatten_window_per_node(X_train_imp), dtype=np.float32)
    Xgb_test  = np.asarray(flatten_window_per_node(X_test_imp),  dtype=np.float32)

    t90 = np.percentile(ytr_all[mtr], 90)
    t95 = np.percentile(ytr_all[mtr], 95)
    t99 = np.percentile(ytr_all[mtr], 99)

    weights = np.ones_like(ytr_all[mtr], dtype=np.float32)
    weights[ytr_all[mtr] >= t90] = w_mid
    weights[ytr_all[mtr] >= t95] = w_danger
    weights[ytr_all[mtr] >= t99] = w_tail

    dtrain = xgb.DMatrix(Xgb_train[mtr], label=res_train[mtr], weight=weights)

    if xgb_params is None:
        xgb_params = {
            "objective": "reg:pseudohubererror",
            "max_depth": 6,
            "eta": 0.05,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "lambda": 1.0,
            "tree_method": "hist",
            "seed": seed,
            "verbosity": 0,
        }

    booster = xgb.train(xgb_params, dtrain, num_boost_round=int(num_boost_round))

    res_test_all = booster.predict(xgb.DMatrix(Xgb_test))
    final_test_all = base["pred_test_all"].copy()
    final_test_all[mte] = final_test_all[mte] + res_test_all[mte]

    return {
        "y_true": base["y_test_all"][mte],
        "y_pred_graph": base["pred_test_all"][mte],
        "y_pred_final": final_test_all[mte],
        "gmodel": base["model"],
        "hmodel": booster
    }

DEFAULT_XGB_PARAMS = {
    "objective": "reg:pseudohubererror",
    "max_depth": 6,
    "eta": 0.05,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "lambda": 1.0,
    "tree_method": "hist",
    "verbosity": 0,
}


def _tail_weights(y_obs, w_mid=2.0, w_danger=5.0, w_tail=10.0):
    t90, t95, t99 = (np.percentile(y_obs, p) for p in (90.0, 95.0, 99.0))
    w = np.ones_like(y_obs, dtype=np.float32)
    w[y_obs >= t90] = w_mid
    w[y_obs >= t95] = w_danger
    w[y_obs >= t99] = w_tail
    return w


def _xgb_params(seed, override=None):
    p = dict(DEFAULT_XGB_PARAMS)
    p["seed"] = seed
    if override:
        p.update(override)
    return p


def _residual_stage(ytr_all, pred_tr_all, mtr, pred_te_all, mte,
                    Xgb_train, Xgb_test, xgb_params=None,
                    num_boost_round=400, seed=42,
                    w_mid=2.0, w_danger=5.0, w_tail=10.0):
    res_train = ytr_all - pred_tr_all
    weights = _tail_weights(ytr_all[mtr], w_mid, w_danger, w_tail)
    dtrain = xgb.DMatrix(Xgb_train[mtr], label=res_train[mtr], weight=weights)
    booster = xgb.train(_xgb_params(seed, xgb_params), dtrain,
                        num_boost_round=int(num_boost_round))
    res_test_all = booster.predict(xgb.DMatrix(Xgb_test))
    final = pred_te_all.copy()
    final[mte] = final[mte] + res_test_all[mte]
    return final, booster


def _flat_design(X_train_imp, X_test_imp):
    return (np.asarray(flatten_window_per_node(X_train_imp), dtype=np.float32),
            np.asarray(flatten_window_per_node(X_test_imp), dtype=np.float32))


def fit_flat_hybrid(model, X_train_imp, Y_train, M_train,
                    X_test_imp, Y_test, M_test, **kw):
    Xtr, Xte = _flat_design(X_train_imp, X_test_imp)
    ytr, yte = Y_train.reshape(-1), Y_test.reshape(-1)
    mtr, mte = M_train.reshape(-1) > 0.5, M_test.reshape(-1) > 0.5

    model.fit(Xtr[mtr], ytr[mtr])
    pred_tr_all = np.asarray(model.predict(Xtr), dtype=np.float64)
    pred_te_all = np.asarray(model.predict(Xte), dtype=np.float64)

    final, booster = _residual_stage(ytr, pred_tr_all, mtr, pred_te_all, mte,
                                     Xtr, Xte, **kw)
    return {"y_true": yte[mte], "y_pred_base": pred_te_all[mte],
            "y_pred_final": final[mte], "gmodel": model, "hmodel": booster}


def fit_static_graph_hybrid(model, X_train_imp, Y_train, M_train,
                            X_test_imp, Y_test, M_test,
                            A_static, tau=0, **kw):
    Ztr = make_graph_features_static(X_train_imp, A_static, tau=tau)
    Zte = make_graph_features_static(X_test_imp, A_static, tau=tau)
    Ztr_f = Ztr.reshape(-1, Ztr.shape[-1])
    Zte_f = Zte.reshape(-1, Zte.shape[-1])
    ytr, yte = Y_train.reshape(-1), Y_test.reshape(-1)
    mtr, mte = M_train.reshape(-1) > 0.5, M_test.reshape(-1) > 0.5

    model.fit(Ztr_f[mtr], ytr[mtr])
    pred_tr_all = np.asarray(model.predict(Ztr_f), dtype=np.float64)
    pred_te_all = np.asarray(model.predict(Zte_f), dtype=np.float64)

    Xtr, Xte = _flat_design(X_train_imp, X_test_imp)
    final, booster = _residual_stage(ytr, pred_tr_all, mtr, pred_te_all, mte,
                                     Xtr, Xte, **kw)
    return {"y_true": yte[mte], "y_pred_base": pred_te_all[mte],
            "y_pred_final": final[mte], "gmodel": model, "hmodel": booster}


def fit_tailweighted_xgb(X_train_imp, Y_train, M_train,
                         X_test_imp, Y_test, M_test,
                         xgb_params=None, num_boost_round=400, seed=42,
                         w_mid=2.0, w_danger=5.0, w_tail=10.0,
                         objective="published"):
    Xtr, Xte = _flat_design(X_train_imp, X_test_imp)
    ytr, yte = Y_train.reshape(-1), Y_test.reshape(-1)
    mtr, mte = M_train.reshape(-1) > 0.5, M_test.reshape(-1) > 0.5

    weights = _tail_weights(ytr[mtr], w_mid, w_danger, w_tail)

    if objective == "published":
        model = XGBRegressor(
            n_estimators=200,
            max_depth=8,
            learning_rate=0.07,
            subsample=0.9,
            colsample_bytree=0.8,
            tree_method="hist",
            n_jobs=-1,
            random_state=seed,
        )
        model.fit(Xtr[mtr], ytr[mtr], sample_weight=weights)
        pred = np.asarray(model.predict(Xte), dtype=np.float64)
        booster = model
    elif objective == "pseudohuber":
        prm = _xgb_params(seed, xgb_params)
        prm["base_score"] = float(np.average(ytr[mtr], weights=weights))
        dtrain = xgb.DMatrix(Xtr[mtr], label=ytr[mtr], weight=weights)
        booster = xgb.train(prm, dtrain, num_boost_round=int(num_boost_round))
        pred = np.asarray(booster.predict(xgb.DMatrix(Xte)), dtype=np.float64)
    else:
        raise ValueError(f"unknown objective {objective!r}")

    const = float(np.average(ytr[mtr], weights=weights))
    const_mae = float(np.mean(np.abs(yte[mte] - const)))
    pred_mae = float(np.mean(np.abs(pred[mte] - yte[mte])))
    diverged = (not np.isfinite(pred_mae)) or pred_mae > 2.0 * const_mae
    if diverged:
        print(f"CONTROL Tail-Weighted XGB ({objective}) DIVERGED")
        print(f"MAE {pred_mae:.3f} vs constant-predictor MAE {const_mae:.3f}")
        print(f"pred mean {pred[mte].mean():.3f}  std {pred[mte].std():.6f}")

    return {"y_true": yte[mte], "y_pred": pred[mte], "model": booster,
            "diverged": bool(diverged), "mae": pred_mae,
            "const_mae": const_mae, "objective": objective}

def choose_best_tau_dynamic(train_pack, val_pack, ctx, tau_list):
    rows = []
    best_tau = None
    best_mae = np.inf
    for tau in tau_list:
        if tau >= train_pack["X"].shape[1]:
            continue
        res = fit_huber_graph(
            train_pack["X"], train_pack["Y"], train_pack["M"],
            val_pack["X"], val_pack["Y"], val_pack["M"],
            ctx=ctx, tau=tau
        )
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau": tau, "val_MAE": mae})
        if mae < best_mae:
            best_mae = mae
            best_tau = tau
    return best_tau, pd.DataFrame(rows)

def choose_best_tau_static(train_pack, val_pack, A_static, tau_list):
    rows = []
    best_tau = None
    best_mae = np.inf
    for tau in tau_list:
        if tau >= train_pack["X"].shape[1]:
            continue
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("huber", HuberRegressor(epsilon=1.35, max_iter=500))
        ])
        res = fit_static_graph_regressor(
            model,
            train_pack["X"], train_pack["Y"], train_pack["M"],
            val_pack["X"], val_pack["Y"], val_pack["M"],
            A_static=A_static, tau=tau
        )
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau": tau, "val_MAE": mae})
        if mae < best_mae:
            best_mae = mae
            best_tau = tau
    return best_tau, pd.DataFrame(rows)

def eval_hybrid_params(params, tau, train_pack, val_pack, ctx):
    alpha, eps, log_huber_alpha, max_depth, eta, subsample, colsample, reg_lambda, nround, w_mid, w_danger, w_tail = params
    huber_alpha = 10 ** float(log_huber_alpha)

    hybrid = fit_huber_hybrid(
        train_pack["X"], train_pack["Y"], train_pack["M"],
        val_pack["X"],   val_pack["Y"],   val_pack["M"],
        ctx=ctx,
        tau=tau,
        alpha=float(alpha),
        eps=float(eps),
        huber_alpha=float(huber_alpha),
        xgb_params={
            "objective": "reg:pseudohubererror",
            "max_depth": int(round(max_depth)),
            "eta": float(eta),
            "subsample": float(subsample),
            "colsample_bytree": float(colsample),
            "lambda": float(reg_lambda),
            "tree_method": "hist",
            "seed": RANDOM_SEED,
            "verbosity": 0,
        },
        num_boost_round=int(round(nround)),
        seed=RANDOM_SEED,
        w_mid=float(w_mid),
        w_danger=float(w_danger),
        w_tail=float(w_tail)
    )

    y_true = hybrid["y_true"]
    y_pred = hybrid["y_pred_final"]
    mae = mean_absolute_error(y_true, y_pred)

    thr99 = np.percentile(y_true, 99)
    idx_tail = y_true >= thr99
    if idx_tail.sum() > 0:
        mfb_tail, _ = compute_mfb_nmse(y_true[idx_tail], y_pred[idx_tail])
        return mae + (10.0 * abs(mfb_tail))
    return mae

def sample_population(n, bounds):
    pop = np.random.rand(n, bounds.shape[0])
    return bounds[:, 0] + pop * (bounds[:, 1] - bounds[:, 0])

def clip_params(x, bounds):
    return np.minimum(np.maximum(x, bounds[:, 0]), bounds[:, 1])

JSO_TRACE = []


def eval_hybrid_params_ctrl(params, train_pack, val_pack, variant,
                            A_static=None, tau=0):
    (log_huber_alpha, max_depth, eta, subsample, colsample,
     reg_lambda, nround, w_mid, w_danger, w_tail) = params
    huber_alpha = 10 ** float(log_huber_alpha)

    base = Pipeline([("scaler", StandardScaler()),
                     ("huber", HuberRegressor(epsilon=1.35,
                                              alpha=float(huber_alpha),
                                              max_iter=500))])
    kw = dict(
        xgb_params={
            "objective": "reg:pseudohubererror",
            "max_depth": int(round(max_depth)),
            "eta": float(eta),
            "subsample": float(subsample),
            "colsample_bytree": float(colsample),
            "lambda": float(reg_lambda),
            "tree_method": "hist",
            "seed": RANDOM_SEED,
            "verbosity": 0,
        },
        num_boost_round=int(round(nround)),
        seed=RANDOM_SEED,
        w_mid=float(w_mid), w_danger=float(w_danger), w_tail=float(w_tail),
    )
    args = (train_pack["X"], train_pack["Y"], train_pack["M"],
            val_pack["X"],   val_pack["Y"],   val_pack["M"])
    if variant == "flat":
        r = fit_flat_hybrid(base, *args, **kw)
    elif variant == "static":
        assert A_static is not None
        r = fit_static_graph_hybrid(base, *args, A_static=A_static,
                                    tau=int(tau), **kw)
    else:
        raise ValueError(f"unknown control variant: {variant!r}")

    y_true, y_pred = r["y_true"], r["y_pred_final"]
    if not np.isfinite(y_pred).all():
        return float("inf")
    mae = mean_absolute_error(y_true, y_pred)
    thr99 = np.percentile(y_true, 99)
    idx = y_true >= thr99
    if idx.sum() > 0:
        mfb_tail, _ = compute_mfb_nmse(y_true[idx], y_pred[idx])
        return mae + (10.0 * abs(mfb_tail))
    return mae


def jso_optimize(train_pack, val_pack, ctx, tau, bounds, n_pop=12, iters=15,
                 seed=1, fitness_fn=None, label="dynamic"):
    if fitness_fn is None:
        def fitness_fn(p):
            return eval_hybrid_params(p, tau, train_pack, val_pack, ctx)
    np.random.seed(seed)
    dim = bounds.shape[0]
    pop = sample_population(n_pop, bounds)
    fitness = np.array([fitness_fn(p) for p in pop], dtype=float)
    _n_eval = int(n_pop)

    def _trace(it):
        JSO_TRACE.append(dict(
            label=label, seed=int(seed), dim=int(dim), n_pop=int(n_pop),
            iter=int(it), n_eval=int(_n_eval),
            best=float(np.min(fitness)), worst=float(np.max(fitness)),
            mean=float(np.mean(fitness)), sd=float(np.std(fitness, ddof=0))))
    _trace(0)

    best_idx = np.argmin(fitness)
    best_p = pop[best_idx].copy()
    best_f = float(fitness[best_idx])
    print(f"    initial best objective={best_f:.4f}")

    for it in range(iters):
        c = (1.0 - it / max(iters, 1))
        new_pop = pop.copy()
        for i in range(n_pop):
            if np.random.rand() < 0.5:
                step = np.random.randn(dim) * c * 0.1
                cand = pop[i] + step + c * (best_p - pop[i]) * np.random.rand(dim)
            else:
                j = np.random.randint(0, n_pop)
                step = (pop[j] - pop[i]) * (np.random.rand(dim) - 0.5) * c
                cand = pop[i] + step
            new_pop[i] = clip_params(cand, bounds)

        new_fit = np.array([fitness_fn(p) for p in new_pop], dtype=float)
        _n_eval += int(n_pop)
        improved = new_fit < fitness
        pop[improved] = new_pop[improved]
        fitness[improved] = new_fit[improved]

        best_idx = np.argmin(fitness)
        if fitness[best_idx] < best_f:
            best_f = float(fitness[best_idx])
            best_p = pop[best_idx].copy()

        _trace(it + 1)
        print(f"    iter {it + 1:02d}/{iters:02d}: best objective={best_f:.4f}")

    print(f"    [{label}] dim={dim} evaluations={_n_eval} "
          f"best={best_f:.4f}")
    return best_p, best_f

def add_result_row(rows, tail_rows, H, split_name, model_name, y_true, y_pred, extra=None):
    summary = summarize_regression(y_true, y_pred)
    row = {
        "H": H,
        "split": split_name,
        "model": model_name,
        "MAE": summary["MAE"],
        "RMSE": summary["RMSE"],
        "R2": summary["R2"],
        "n_test": int(len(y_true))
    }
    if extra:
        row.update(extra)
    rows.append(row)

    for p, thr, mae, rmse, n_tail in compute_tail_metrics(y_true, y_pred):
        tail_row = {
            "H": H,
            "split": split_name,
            "model": model_name,
            "percentile": p,
            "threshold": float(thr),
            "tail_MAE": float(mae) if pd.notna(mae) else np.nan,
            "tail_RMSE": float(rmse) if pd.notna(rmse) else np.nan,
            "n_tail": int(n_tail)
        }
        if extra:
            tail_row.update(extra)
        tail_rows.append(tail_row)

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH).copy()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["station_id"] = df["station_id"].astype(str)

stations = (
    df[["station_id", "station_name"]]
    .drop_duplicates()
    .sort_values("station_id")
    .reset_index(drop=True)
)
station_ids = stations["station_id"].tolist()
N = len(station_ids)

candidate_features = ["pm25", "temp_2m", "dewpoint_2m", "surface_pressure", "u10", "v10"]
features = [c for c in candidate_features if c in df.columns]
assert "pm25" in features
pm25_idx = features.index("pm25")
u_idx = features.index("u10")
v_idx = features.index("v10")

all_times = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")
base = pd.MultiIndex.from_product([all_times, station_ids], names=["timestamp", "station_id"]).to_frame(index=False)

aligned = base.merge(
    df[["timestamp", "station_id"] + features],
    on=["timestamp", "station_id"],
    how="left"
)

X_feat = []
for feat in features:
    mat = aligned.pivot(index="timestamp", columns="station_id", values=feat).reindex(all_times)[station_ids]
    X_feat.append(mat.to_numpy(dtype=np.float32))

X_all_raw = np.stack(X_feat, axis=-1).astype(np.float32)

pm25_raw = aligned.pivot(index="timestamp", columns="station_id", values="pm25").reindex(all_times)[station_ids]
pm25_raw_np = pm25_raw.to_numpy(dtype=np.float32)
Y_mask_full = (~np.isnan(pm25_raw_np)).astype(np.float32)

train_timeline_mask = all_times < SPLIT_TIME
fill_values = build_fill_values_from_train_timeline(X_all_raw[train_timeline_mask])

nodes, edges_df = build_nodes_edges(df, station_ids, k=GRAPH_K)
A_static = build_static_adj_from_nodes(nodes, k=GRAPH_K, sigma_km=None, self_loops=False, row_normalize=True)

src = edges_df["src"].to_numpy(dtype=int)
dst = edges_df["dst"].to_numpy(dtype=int)
w_dist = edges_df["w_dist"].to_numpy(dtype=np.float32)

src_lat = nodes.loc[src, "lat"].to_numpy()
src_lon = nodes.loc[src, "lon"].to_numpy()
dst_lat = nodes.loc[dst, "lat"].to_numpy()
dst_lon = nodes.loc[dst, "lon"].to_numpy()
edge_bearing = bearing_radians(src_lat, src_lon, dst_lat, dst_lon)

_rng_shuf = np.random.RandomState(12345)
edge_bearing_shuffled = edge_bearing[_rng_shuf.permutation(len(edge_bearing))]

graph_ctx = {
    "src": src,
    "dst": dst,
    "w_dist": w_dist,
    "edge_bearing": edge_bearing,
    "edge_bearing_shuffled": edge_bearing_shuffled,
    "u_idx": u_idx,
    "v_idx": v_idx,
    "N": len(nodes),
}
print(f"graph_ctx ready | GATE_MODE={GATE_MODE!r} | "
      f"{len(src)} edges over {len(nodes)} nodes")

graph_ctx ready | GATE_MODE='fixed' | 32 edges over 8 nodes


In [ ]:
# CONTROLS PASS

import json
import time

os.makedirs(CTRL_CKPT_DIR, exist_ok=True)
_ctrl_t0 = time.time()

print("=" * 74)
print("CONTROLS-ONLY PASS")
print("=" * 74)
print(f"  cache dir  : {CTRL_CACHE_DIR}")
print(f"  checkpoints: {CTRL_CKPT_DIR}")
print(f"  horizons   : {CTRL_H_LIST}")
print(f"  JSO budget : pop={JSO_POP} iters={JSO_ITERS}: "
      f"{JSO_POP * (JSO_ITERS + 1)} evals per control per H")
print(f"  gate       : GATE_MODE={GATE_MODE!r}")

if CTRL_TAU_CSV and os.path.exists(CTRL_TAU_CSV):
    _t = pd.read_csv(CTRL_TAU_CSV)
    _col = "tau_static" if "tau_static" in _t.columns else None
    if _col:
        CTRL_TAU_STATIC = {int(r.H): int(r[_col]) for _, r in _t.iterrows()}
        print(f"  tau_static : loaded from {CTRL_TAU_CSV}: {CTRL_TAU_STATIC}")


def _ctrl_load_h(H):
    d = os.path.join(CTRL_CACHE_DIR, f"H{H}")
    need = ["X_train_imp", "Y_train", "M_train",
            "X_test_imp", "Y_test", "M_test"]
    miss = [k for k in need if not os.path.exists(os.path.join(d, k + ".npy"))]
    assert not miss
    out = {k: np.load(os.path.join(d, k + ".npy")) for k in need}
    print(f"  H={H:<3d} loaded cached tensors  train={out['X_train_imp'].shape}"
          f"  test={out['X_test_imp'].shape}")
    return out


def _ctrl_prep(variant, Xtr, Ytr, Mtr, Xte, Yte, Mte, tau_static):
    res_tr, res_te = _flat_design(Xtr, Xte)
    if variant == "flat":
        base_tr, base_te = res_tr, res_te
    elif variant == "static":
        Ztr = make_graph_features_static(Xtr, A_static, tau=int(tau_static))
        Zte = make_graph_features_static(Xte, A_static, tau=int(tau_static))
        base_tr = Ztr.reshape(-1, Ztr.shape[-1])
        base_te = Zte.reshape(-1, Zte.shape[-1])
    else:
        raise ValueError(variant)
    return {"base_tr": base_tr, "base_te": base_te,
            "res_tr": res_tr, "res_te": res_te,
            "ytr": Ytr.reshape(-1), "yte": Yte.reshape(-1),
            "mtr": Mtr.reshape(-1) > 0.5, "mte": Mte.reshape(-1) > 0.5}


def _ctrl_fitness_fast(params, pre):
    (lha, md, eta, sub, col, lam, nr, wm, wd, wt) = params
    ha = 10 ** float(lha)
    base = Pipeline([("scaler", StandardScaler()),
                     ("huber", HuberRegressor(epsilon=1.35, alpha=float(ha),
                                              max_iter=500))])
    Btr, Bte = pre["base_tr"], pre["base_te"]
    ytr, yte = pre["ytr"], pre["yte"]
    mtr, mte = pre["mtr"], pre["mte"]
    base.fit(Btr[mtr], ytr[mtr])
    ptr = np.asarray(base.predict(Btr), dtype=np.float64)
    pte = np.asarray(base.predict(Bte), dtype=np.float64)
    final, _ = _residual_stage(
        ytr, ptr, mtr, pte, mte, pre["res_tr"], pre["res_te"],
        xgb_params={"objective": "reg:pseudohubererror",
                    "max_depth": int(round(md)), "eta": float(eta),
                    "subsample": float(sub), "colsample_bytree": float(col),
                    "lambda": float(lam), "tree_method": "hist",
                    "seed": RANDOM_SEED, "verbosity": 0},
        num_boost_round=int(round(nr)), seed=RANDOM_SEED,
        w_mid=float(wm), w_danger=float(wd), w_tail=float(wt))
    y_true, y_pred = yte[mte], final[mte]
    if not np.isfinite(y_pred).all():
        return float("inf")
    mae = mean_absolute_error(y_true, y_pred)
    thr99 = np.percentile(y_true, 99)
    idx = y_true >= thr99
    if idx.sum() > 0:
        mfb_tail, _ = compute_mfb_nmse(y_true[idx], y_pred[idx])
        return mae + (10.0 * abs(mfb_tail))
    return mae


_ctrl_main, _ctrl_tail, _ctrl_hp = [], [], []
_bounds_ctrl = bounds[2:, :]
assert _bounds_ctrl.shape[0] == bounds.shape[0] - 2 == 10
_PIECE_LABEL = {"xgb": "Tail-Weighted XGB (both objectives)",
                "flat": "CONTROL Tuned Flat Hybrid (no graph)",
                "static": "CONTROL Tuned Static-Graph Hybrid"}
_EXTRA_PIECES = ("jsoseed", "randsearch", "ksens")
for _p in CTRL_PIECES:
    assert _p in _PIECE_LABEL or _p in _EXTRA_PIECES


def _ckp(H, piece, kind):
    return os.path.join(CTRL_CKPT_DIR, f"H{H}_{piece}_{kind}.csv")


def _piece_done(H, piece):
    if CTRL_FORCE:
        return False
    if all(os.path.exists(_ckp(H, piece, k)) for k in ("main", "tail")):
        return True
    _old = [os.path.join(CTRL_CKPT_DIR, f"H{H}_{k}.csv") for k in ("main", "tail")]
    return all(os.path.exists(p) for p in _old)


def _ctrl_env_stamp(H, Xte, Yte, Mte):
    import hashlib, platform, sklearn, xgboost

    def _sha(a):
        return hashlib.sha256(
            np.ascontiguousarray(a).tobytes()).hexdigest()[:16]

    return {"H": int(H),
            "x_test_sha": _sha(Xte), "y_test_sha": _sha(Yte),
            "m_test_sha": _sha(Mte),
            "x_test_shape": str(tuple(Xte.shape)),
            "n_test_masked": int((Mte.reshape(-1) > 0.5).sum()),
            "python": platform.python_version(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
            "random_seed": int(RANDOM_SEED), "jso_pop": int(JSO_POP),
            "jso_iters": int(JSO_ITERS), "valid_frac": float(VALID_FRAC),
            "gate_mode": str(GATE_MODE)}


def _write_piece(H, piece, rows, tails, env=None):
    pd.DataFrame(rows).to_csv(_ckp(H, piece, "main"), index=False)
    pd.DataFrame(tails).to_csv(_ckp(H, piece, "tail"), index=False)
    _tr = globals().get("JSO_TRACE")
    if _tr:
        pd.DataFrame(_tr).to_csv(_ckp(H, piece, "jso"), index=False)
    if env is not None:
        json.dump(dict(env, piece=piece),
                  open(_ckp(H, piece, "env").replace(".csv", ".json"), "w"),
                  indent=1, sort_keys=True)
    print(f"  CHECKPOINT  H={H} piece={piece!r}  saved "
          f"{os.path.basename(_ckp(H, piece, 'main'))} "
          f"({len(rows)} model row(s))")
    _ctrl_main.extend(rows)
    _ctrl_tail.extend(tails)


for H in CTRL_H_LIST:
    _todo = [p for p in CTRL_PIECES if not _piece_done(H, p)]
    _skip = [p for p in CTRL_PIECES if p not in _todo]
    if _skip:
        print(f"\nH={H}: already checkpointed, skipping {_skip}")
    if not _todo:
        continue

    _th = time.time()
    print("\n" + "=" * 74)
    print(f"CONTROLS  H={H}   pieces to run: {_todo}")
    print("=" * 74)
    _p = _ctrl_load_h(H)
    Xtr, Ytr, Mtr = _p["X_train_imp"], _p["Y_train"], _p["M_train"]
    Xte, Yte, Mte = _p["X_test_imp"],  _p["Y_test"],  _p["M_test"]
    _env = _ctrl_env_stamp(H, Xte, Yte, Mte)
    print(f"  env: xgboost={_env['xgboost']} sklearn={_env['sklearn']} "
          f"numpy={_env['numpy']} | data sha y={_env['y_test_sha']} "
          f"x={_env['x_test_sha']} n_masked={_env['n_test_masked']}")

    if "xgb" in _todo:
        _rows, _tails = [], []
        for _obj, _lab in (("published",  "CONTROL Tail-Weighted XGB"),
                           ("pseudohuber",
                            "CONTROL Tail-Weighted XGB (pseudo-Huber)")):
            _r = fit_tailweighted_xgb(Xtr, Ytr, Mtr, Xte, Yte, Mte,
                                      objective=_obj, num_boost_round=400,
                                      seed=RANDOM_SEED, w_mid=2.0,
                                      w_danger=5.0, w_tail=10.0)
            _nm = _lab + (" [DIVERGED]" if _r["diverged"] else "")
            add_result_row(_rows, _tails, H, "test", _nm,
                           _r["y_true"], _r["y_pred"])
            print(f"  {_nm}: MAE={_r['mae']:.4f}  "
                  f"constant-predictor MAE={_r['const_mae']:.4f}  "
                  f"diverged={_r['diverged']}")
        _write_piece(H, "xgb", _rows, _tails, env=_env)

    _vars = [v for v in ("flat", "static") if v in _todo]
    if not _vars:
        print(f"  H={H}: no JSO search requested in this job "
              f"({time.time()-_th:.0f}s)")
        continue

    _tv, _vv = train_val_split_time_order(Xtr, Ytr, Mtr,
                                          np.arange(Xtr.shape[0]),
                                          frac=VALID_FRAC)

    _tau_s = 0
    if "static" in _vars:
        if H in CTRL_TAU_STATIC:
            _tau_s = int(CTRL_TAU_STATIC[H])
            print(f"  tau_static={_tau_s} (supplied)")
        else:
            _tau_s, _ = choose_best_tau_static(_tv, _vv, A_static, TAU_LIST)
            print(f"  tau_static={_tau_s} (searched)")

    for _cv in _vars:
        _cname = _PIECE_LABEL[_cv]
        _ctau = _tau_s if _cv == "static" else 0
        print(f"\n  --- {_cname} ---")
        print(f"      budget {JSO_POP} x ({JSO_ITERS}+1) = "
              f"{JSO_POP * (JSO_ITERS + 1)} hybrid fits")
        _t1 = time.time()
        _pre = _ctrl_prep(_cv, _tv["X"], _tv["Y"], _tv["M"],
                          _vv["X"], _vv["Y"], _vv["M"], _ctau)
        print(f"      designs built once in {time.time()-_t1:.1f}s "
              f"(base {_pre['base_tr'].shape}, resid {_pre['res_tr'].shape})")

        if CTRL_VERIFY_FASTPATH:
            _rng = np.random.RandomState(0)
            _pt = _bounds_ctrl[:, 0] + _rng.rand(10) * (
                _bounds_ctrl[:, 1] - _bounds_ctrl[:, 0])
            _pt[6] = 120.0
            _fast = _ctrl_fitness_fast(_pt, _pre)
            _ref = eval_hybrid_params_ctrl(_pt, _tv, _vv, _cv,
                                           A_static=A_static, tau=_ctau)
            _d = abs(_fast - _ref)
            print(f"      fast-path check: fast={_fast:.10f} "
                  f"ref={_ref:.10f}  |diff|={_d:.3e}")
            assert _d <= 1e-8 * max(1.0, abs(_ref))

        _bp, _bo = jso_optimize(
            train_pack=_tv, val_pack=_vv, ctx=graph_ctx, tau=_ctau,
            bounds=_bounds_ctrl, n_pop=JSO_POP, iters=JSO_ITERS,
            seed=RANDOM_SEED, label=f"{_cv}_H{H}",
            fitness_fn=(lambda p, _pp=_pre: _ctrl_fitness_fast(p, _pp)))

        (_lha, _md, _eta, _sub, _col, _lam, _nr, _wm, _wd, _wt) = _bp
        _ha = 10 ** float(_lha)
        _base = Pipeline([("scaler", StandardScaler()),
                          ("huber", HuberRegressor(epsilon=1.35,
                                                   alpha=float(_ha),
                                                   max_iter=500))])
        _kw = dict(xgb_params={"objective": "reg:pseudohubererror",
                               "max_depth": int(round(_md)),
                               "eta": float(_eta), "subsample": float(_sub),
                               "colsample_bytree": float(_col),
                               "lambda": float(_lam),
                               "tree_method": "hist",
                               "seed": RANDOM_SEED, "verbosity": 0},
                   num_boost_round=int(round(_nr)), seed=RANDOM_SEED,
                   w_mid=float(_wm), w_danger=float(_wd), w_tail=float(_wt))
        _args = (Xtr, Ytr, Mtr, Xte, Yte, Mte)
        if _cv == "flat":
            _r = fit_flat_hybrid(_base, *_args, **_kw)
        else:
            _r = fit_static_graph_hybrid(_base, *_args, A_static=A_static,
                                         tau=int(_ctau), **_kw)
        _extra = {"tau": int(_ctau), "huber_alpha": float(_ha),
                  "max_depth": int(round(_md)), "eta": float(_eta),
                  "subsample": float(_sub), "colsample_bytree": float(_col),
                  "lambda": float(_lam), "num_boost_round": int(round(_nr)),
                  "w_mid": float(_wm), "w_danger": float(_wd),
                  "w_tail": float(_wt), "val_objective": float(_bo),
                  "jso_dim": int(_bounds_ctrl.shape[0]),
                  "jso_evals": int(JSO_POP * (JSO_ITERS + 1))}
        _rows, _tails = [], []
        add_result_row(_rows, _tails, H, "test", _cname,
                       _r["y_true"], _r["y_pred_final"], extra=_extra)
        _ctrl_hp.append(dict(H=H, control=_cv, **_extra))
        print(f"  {_cname} done  val_obj={_bo:.4f}  "
              f"({(time.time()-_t1)/3600:.2f} h)")
        _write_piece(H, _cv, _rows, _tails, env=_env)

    if _ctrl_hp:
        pd.DataFrame(_ctrl_hp).to_csv(
            os.path.join(CTRL_CKPT_DIR,
                         f"H{H}_{'_'.join(_vars)}_hyperparams.csv"),
            index=False)
    print(f"\n  H={H} finished this job in {(time.time()-_th)/3600:.2f} h")


def _extra_done(H, piece):
    p = os.path.join(CTRL_CKPT_DIR, f"H{H}_{piece}_extra.csv")
    return (not CTRL_FORCE) and os.path.exists(p)


def _write_extra(H, piece, rows, tails=None, env=None, extra_tag=""):
    pd.DataFrame(rows).to_csv(
        os.path.join(CTRL_CKPT_DIR, f"H{H}_{piece}_extra.csv"), index=False)
    if tails:
        pd.DataFrame(tails).to_csv(
            os.path.join(CTRL_CKPT_DIR, f"H{H}_{piece}_extra_tail.csv"),
            index=False)
    _tr = globals().get("JSO_TRACE")
    if _tr:
        pd.DataFrame(_tr).to_csv(
            os.path.join(CTRL_CKPT_DIR, f"H{H}_{piece}_extra_trace.csv"),
            index=False)
    if env is not None:
        json.dump(dict(env, piece=piece),
                  open(os.path.join(CTRL_CKPT_DIR,
                                    f"H{H}_{piece}_extra_env.json"), "w"),
                  indent=1, sort_keys=True)
    print(f"  CHECKPOINT  H={H} piece={piece!r}{extra_tag}: "
          f"H{H}_{piece}_extra.csv ({len(rows)} row(s))")


def _fit_dynamic_tuned(p12, Xtr, Ytr, Mtr, Xte, Yte, Mte, tau_dyn):
    (al, ep, lha, md, eta, sub, col, lam, nr, wm, wd, wt) = p12
    ha = 10 ** float(lha)
    return fit_huber_hybrid(
        Xtr, Ytr, Mtr, Xte, Yte, Mte, ctx=graph_ctx, tau=int(tau_dyn),
        alpha=float(al), eps=float(ep), huber_alpha=float(ha),
        xgb_params={"objective": "reg:pseudohubererror",
                    "max_depth": int(round(md)), "eta": float(eta),
                    "subsample": float(sub), "colsample_bytree": float(col),
                    "lambda": float(lam), "tree_method": "hist",
                    "seed": RANDOM_SEED, "verbosity": 0},
        num_boost_round=int(round(nr)), seed=RANDOM_SEED,
        w_mid=float(wm), w_danger=float(wd), w_tail=float(wt)), ha


_EXTRA = [p for p in CTRL_PIECES if p in ("jsoseed", "randsearch", "ksens")]
for H in [h for h in CTRL_H_LIST if _EXTRA]:
    _p = None
    for _piece in _EXTRA:
        if _extra_done(H, _piece):
            print(f"\nH={H}: {_piece!r} already checkpointed, skipping")
            continue
        if _p is None:
            print("\n" + "=" * 74)
            print(f"EXTRA PIECES  H={H}: {_EXTRA}")
            print("=" * 74)
            _p = _ctrl_load_h(H)
            Xtr, Ytr, Mtr = _p["X_train_imp"], _p["Y_train"], _p["M_train"]
            Xte, Yte, Mte = _p["X_test_imp"],  _p["Y_test"],  _p["M_test"]
            _env = _ctrl_env_stamp(H, Xte, Yte, Mte)
            _tv, _vv = train_val_split_time_order(
                Xtr, Ytr, Mtr, np.arange(Xtr.shape[0]), frac=VALID_FRAC)
        assert H in CTRL_TAU_DYN
        _td = int(CTRL_TAU_DYN[H])
        _t0 = time.time()

        if _piece == "jsoseed":
            JSO_TRACE.clear()
            rows, tails = [], []
            for _s in CTRL_JSO_SEEDS:
                print(f"\n  --- JSO seed {_s} (dynamic graph, tau={_td}, "
                      f"budget {JSO_POP}x({JSO_ITERS}+1)) ---")
                _ts = time.time()
                _bp, _bo = jso_optimize(
                    train_pack=_tv, val_pack=_vv, ctx=graph_ctx, tau=_td,
                    bounds=bounds, n_pop=JSO_POP, iters=JSO_ITERS, seed=_s,
                    label=f"jso_seed{_s}_H{H}")
                _r, _ha = _fit_dynamic_tuned(_bp, Xtr, Ytr, Mtr,
                                             Xte, Yte, Mte, _td)
                _nm = f"Tuned Hybrid (JSO seed {_s})"
                add_result_row(rows, tails, H, "test", _nm,
                               _r["y_true"], _r["y_pred_final"],
                               extra={"jso_seed": int(_s), "tau": _td,
                                      "val_objective": float(_bo),
                                      "huber_alpha": float(_ha),
                                      "jso_evals": int(JSO_POP*(JSO_ITERS+1)),
                                      "search": "JSO"})
                print(f"  seed {_s}: val_obj={_bo:.6f}  test MAE="
                      f"{rows[-1]['MAE']:.4f}  ({(time.time()-_ts)/3600:.2f} h)")
            _vo = [r["val_objective"] for r in rows]
            print(f"\n  summary over {len(_vo)} extra seed(s): "
                  f"mean={np.mean(_vo):.6f} sd={np.std(_vo, ddof=1):.6f} "
                  f"best={np.min(_vo):.6f} worst={np.max(_vo):.6f}")
            _write_extra(H, _piece, rows, tails, _env)

        elif _piece == "randsearch":
            _nev = int(JSO_POP * (JSO_ITERS + 1))
            print(f"\n  --- random search, {_nev} evaluations, same 12-dim "
                  f"space, same fitness (tau={_td}) ---")
            JSO_TRACE.clear()
            np.random.seed(RANDOM_SEED)
            _cand = sample_population(_nev, bounds)
            _best_f, _best_p, _hist = float("inf"), None, []
            for _i, _c in enumerate(_cand):
                _f = eval_hybrid_params(_c, _td, _tv, _vv, graph_ctx)
                if _f < _best_f:
                    _best_f, _best_p = float(_f), _c.copy()
                _hist.append({"eval": _i + 1, "fitness": float(_f),
                              "best_so_far": _best_f})
                if (_i + 1) % 24 == 0:
                    print(f"      {_i+1:4d}/{_nev} evaluations, "
                          f"best={_best_f:.6f}")
            pd.DataFrame(_hist).to_csv(
                os.path.join(CTRL_CKPT_DIR,
                             f"H{H}_randsearch_extra_history.csv"), index=False)
            _fv = np.array([h["fitness"] for h in _hist], dtype=float)
            _fv = _fv[np.isfinite(_fv)]
            print(f"\n  random search: best={_best_f:.6f}  "
                  f"mean={_fv.mean():.6f}  sd={_fv.std(ddof=1):.6f}  "
                  f"worst={_fv.max():.6f}  (n_finite={len(_fv)})")
            _r, _ha = _fit_dynamic_tuned(_best_p, Xtr, Ytr, Mtr,
                                         Xte, Yte, Mte, _td)
            rows, tails = [], []
            add_result_row(rows, tails, H, "test",
                           "Tuned Hybrid (random search)",
                           _r["y_true"], _r["y_pred_final"],
                           extra={"tau": _td, "val_objective": float(_best_f),
                                  "huber_alpha": float(_ha),
                                  "jso_evals": _nev, "search": "random"})
            print(f"  test MAE={rows[-1]['MAE']:.4f}")
            _write_extra(H, _piece, rows, tails, _env)

        elif _piece == "ksens":
            rows, tails = [], []
            for _k in CTRL_K_LIST:
                print(f"\n  --- k={_k} (untuned Huber-Graph + Hybrid) ---")
                _nd, _ed = build_nodes_edges(df, station_ids, k=int(_k))
                _As = build_static_adj_from_nodes(_nd, k=int(_k),
                                                  sigma_km=None,
                                                  self_loops=False,
                                                  row_normalize=True)
                _s_, _d_ = (_ed["src"].to_numpy(dtype=int),
                            _ed["dst"].to_numpy(dtype=int))
                _eb = bearing_radians(_nd.loc[_s_, "lat"].to_numpy(),
                                      _nd.loc[_s_, "lon"].to_numpy(),
                                      _nd.loc[_d_, "lat"].to_numpy(),
                                      _nd.loc[_d_, "lon"].to_numpy())
                _rs = np.random.RandomState(12345)
                _ctx = {"src": _s_, "dst": _d_,
                        "w_dist": _ed["w_dist"].to_numpy(dtype=np.float32),
                        "edge_bearing": _eb,
                        "edge_bearing_shuffled": _eb[_rs.permutation(len(_eb))],
                        "u_idx": graph_ctx["u_idx"],
                        "v_idx": graph_ctx["v_idx"], "N": graph_ctx["N"]}
                _bt, _tt = choose_best_tau_dynamic(_tv, _vv, _ctx, TAU_LIST)
                print(f"      {len(_ed)} directed edges, tau_dyn={_bt}")
                _g = fit_huber_graph(Xtr, Ytr, Mtr, Xte, Yte, Mte,
                                     ctx=_ctx, tau=_bt)
                add_result_row(rows, tails, H, "test",
                               f"Huber Graph (k={_k})",
                               _g["y_true"], _g["y_pred"],
                               extra={"k": int(_k), "tau": int(_bt),
                                      "n_edges": int(len(_ed))})
                _h = fit_huber_hybrid(Xtr, Ytr, Mtr, Xte, Yte, Mte,
                                      ctx=_ctx, tau=_bt,
                                      num_boost_round=400, seed=RANDOM_SEED)
                add_result_row(rows, tails, H, "test",
                               f"Hybrid (k={_k})",
                               _h["y_true"], _h["y_pred_final"],
                               extra={"k": int(_k), "tau": int(_bt),
                                      "n_edges": int(len(_ed))})
                print(f"      Huber-Graph MAE={rows[-2]['MAE']:.4f}   "
                      f"Hybrid MAE={rows[-1]['MAE']:.4f}")
            _write_extra(H, _piece, rows, tails, _env)

        print(f"  {_piece!r} for H={H} took {(time.time()-_t0)/3600:.2f} h")


_srcs = [CTRL_CKPT_DIR] + [d for d in CTRL_MERGE_DIRS if d]
if CTRL_AUTOMERGE and os.path.isdir("/kaggle/input"):
    for _root, _dirs, _ in os.walk("/kaggle/input"):
        if _root.rstrip("/").count("/") - 2 >= 5:
            _dirs[:] = []
            continue
        for _d in list(_dirs):
            if _d == "controls_ckpt":
                _p = os.path.join(_root, _d)
                if _p not in _srcs:
                    _srcs.append(_p)
_seen, _uniq = set(), []
for _d in _srcs:
    _r = os.path.realpath(_d)
    if os.path.isdir(_d) and _r not in _seen:
        _seen.add(_r)
        _uniq.append(_d)

print("\n" + "=" * 74)
print("ASSEMBLING CHECKPOINTS")
print("=" * 74)
_mparts, _tparts, _prov = [], [], []
for _d in _uniq:
    _mf = sorted(f for f in os.listdir(_d) if f.endswith("_main.csv"))
    _tf = sorted(f for f in os.listdir(_d) if f.endswith("_tail.csv"))
    _hs = sorted(int(f[1:].split("_")[0]) for f in _mf)
    tag = "this session" if os.path.realpath(_d) == os.path.realpath(CTRL_CKPT_DIR) else "attached"
    print(f"  {tag:12s} {_d}\n               horizons {_hs}")
    for f in _mf:
        _mparts.append(pd.read_csv(os.path.join(_d, f)))
    for f in _tf:
        _tparts.append(pd.read_csv(os.path.join(_d, f)))
    _prov += [(h, _d) for h in _hs]

if _mparts:
    controls_main_df = (pd.concat(_mparts, ignore_index=True)
                        .drop_duplicates(subset=["H", "model"], keep="first")
                        .sort_values(["H", "model"]).reset_index(drop=True))
    controls_tail_df = (pd.concat(_tparts, ignore_index=True)
                        .drop_duplicates(subset=["H", "model", "percentile"],
                                         keep="first")
                        .sort_values(["H", "model", "percentile"])
                        .reset_index(drop=True))
    controls_main_df.to_csv(os.path.join(OUT_DIR, "controls_results_main.csv"),
                            index=False)
    controls_tail_df.to_csv(os.path.join(OUT_DIR, "controls_results_tail.csv"),
                            index=False)
    _done = sorted(controls_main_df.H.unique().tolist())
    _nmod = controls_main_df.groupby("H").model.nunique().to_dict()
    print("\n" + "=" * 74)
    print(f"CONTROLS PRESENT for horizons {_done}")
    print(f"  models per horizon: {_nmod}   (4 expected)")
    print(f"  time this session: {(time.time()-_ctrl_t0)/60:.1f} min")
    print(f"  saved controls_results_main.csv  ({len(controls_main_df)} rows)")
    print(f"  saved controls_results_tail.csv  ({len(controls_tail_df)} rows)")
    _short = {h: n for h, n in _nmod.items() if n != 4}
    if _short:
        print(f"  WARNING: horizons with != 4 control rows: {_short}")
    _todo = [h for h in H_LIST if h not in _done]
    if _todo:
        print(f"\n  STILL TO DO: horizons {_todo}")
        print(f"  next run: CTRL_H_LIST = {_todo}")
    else:
        print("\n  all five horizons present")
    print("=" * 74)
    display(controls_main_df)
else:
    print("\nno checkpoints found")

In [6]:
import io, os, json, platform, time, gc, pickle
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import sklearn

# CONFIG
FIXED_DIR      = "huber_processed_leakfree_fixed_old"
MODELS_PKL     = os.path.join(FIXED_DIR, "all_models.pkl")
RESULTS_CSV    = os.path.join(FIXED_DIR, "all_results_main.csv")
CTRL_CACHE_DIR = globals().get("CTRL_CACHE_DIR", "huber_processed_leakfree")
OUT_DIR        = globals().get("OUT_DIR", "Timing_tuned")
BENCH_H_LIST   = [1, 3, 6, 12, 24]
BENCH_WINDOWS  = [1, 8, 64]
BENCH_WARMUP   = 20
BENCH_REPEATS  = 200
BENCH_THREADS  = 1
MAE_TOL        = 1e-4
MODEL_KEY      = "tuned_hybrid_graph"
TUNED_ROW      = "Tuned Hybrid (Graph + XGB residual)"

EXPECTED_MAE = {1: 6.292063, 3: 7.485896, 6: 7.999757, 12: 8.056644, 24: 8.255011}
os.makedirs(OUT_DIR, exist_ok=True)
assert globals().get("GATE_MODE", "fixed") == "fixed"
for p in (MODELS_PKL, RESULTS_CSV, CTRL_CACHE_DIR):
    assert os.path.exists(p)

print("=" * 78)
print("DEPLOYMENT BENCHMARK (tuned DGH-XH)")
print("=" * 78)


# environment
def _cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or "unknown"


def _mem_gb():
    try:
        import psutil
        return round(psutil.virtual_memory().total / 2**30, 1)
    except Exception:
        return None


ENV = {
    "processor_model": _cpu_model(),
    "logical_cores": os.cpu_count(),
    "memory_gb": _mem_gb(),
    "os": f"{platform.system()} {platform.release()} ({platform.version()})",
    "python": platform.python_version(),
    "numpy": np.__version__, "pandas": pd.__version__,
    "sklearn": sklearn.__version__, "xgboost": xgb.__version__,
    "blas_thread_env": {k: os.environ.get(k) for k in
                        ("OMP_NUM_THREADS", "MKL_NUM_THREADS",
                         "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS")},
    "xgboost_predict_threads": BENCH_THREADS,
    "warmup_passes": BENCH_WARMUP, "timed_repetitions": BENCH_REPEATS,
    "timer": "time.perf_counter",
    "batch_unit": "forecast cycle = one 24-h window x all stations",
    "note": "inference only; models loaded from disk, no training, no I/O in timed region",
}
for k, v in ENV.items():
    print(f"  {k:24s} {v}")

res = pd.read_csv(RESULTS_CSV)
tuned = (res[(res["split"] == "test") & (res["model"] == TUNED_ROW)]
         .set_index("H").sort_index())
PARAMS = {}
print(f"\n  tuned parameters from {RESULTS_CSV}")
for H in BENCH_H_LIST:
    r = tuned.loc[H]
    PARAMS[H] = dict(tau=int(round(r["tau"])), alpha=float(r["alpha"]),
                     eps=float(r["eps"]), csv_mae=float(r["MAE"]))
    flag = "ok" if abs(r["MAE"] - EXPECTED_MAE[H]) < 1e-5 else \
        "NOT THE FIXED RUN — check FIXED_DIR"
    print(f"    H={H:<2d} tau={PARAMS[H]['tau']}  alpha={PARAMS[H]['alpha']:.6f}  "
          f"eps={PARAMS[H]['eps']:.6f}  csv MAE={r['MAE']:.6f}  [{flag}]")
    assert flag == "ok"

# load models
all_models = joblib.load(MODELS_PKL)


def _unpack(entry):
    if isinstance(entry, dict):
        g = entry.get("gmodel") or entry.get("model")
        h = entry.get("hmodel") or entry.get("resid_model")
    else:
        g, h = entry[0], entry[1]
    assert hasattr(g, "predict") and isinstance(h, xgb.Booster)
    return g, h


MODELS = {}
for H in BENCH_H_LIST:
    g, h = _unpack(all_models[H][MODEL_KEY])
    h.set_param({"nthread": BENCH_THREADS})
    MODELS[H] = (g, h)
del all_models
gc.collect()


def _load_test(H):
    d = os.path.join(CTRL_CACHE_DIR, f"H{H}")
    return {k: np.load(os.path.join(d, k + ".npy"))
            for k in ("X_test_imp", "Y_test", "M_test")}


def predict_tuned(H, Xw):
    g, h = MODELS[H]
    p = PARAMS[H]
    Z = make_graph_features_dynamic(Xw, graph_ctx, tau=p["tau"], alpha=p["alpha"],
                                    eps=p["eps"], gate_mode="fixed")
    base = g.predict(Z.reshape(-1, Z.shape[-1]))
    flat = np.asarray(flatten_window_per_node(Xw), dtype=np.float32)
    return base + h.predict(xgb.DMatrix(flat))


print("\n  MAE check")
TEST = {}
for H in BENCH_H_LIST:
    t = _load_test(H)
    TEST[H] = t["X_test_imp"]
    m = t["M_test"].reshape(-1) > 0.5
    y = t["Y_test"].reshape(-1)
    mae = float(np.mean(np.abs(predict_tuned(H, t["X_test_imp"])[m] - y[m])))
    d = mae - PARAMS[H]["csv_mae"]
    ok = abs(d) <= MAE_TOL
    print(f"    H={H:<2d} reproduced {mae:.6f}  csv {PARAMS[H]['csv_mae']:.6f}  "
          f"diff {d:+.2e}  {'PASS' if ok else 'FAIL'}")
    assert ok
N_ST = TEST[1].shape[2]
print(f"  all horizons ok, stations = {N_ST}")


# timing
def _timeit(fn):
    for _ in range(BENCH_WARMUP):
        fn()
    gc.collect()
    out = np.empty(BENCH_REPEATS)
    for i in range(BENCH_REPEATS):
        t0 = time.perf_counter()
        fn()
        out[i] = (time.perf_counter() - t0) * 1e3
    return out


rows = []


def _record(H, stage, nw, t):
    rows.append(dict(H=H, stage=stage, windows=nw, station_hours=nw * N_ST,
                     mean_ms=t.mean(), sd_ms=t.std(ddof=1), median_ms=np.median(t),
                     p95_ms=np.percentile(t, 95),
                     per_station_hour_us=t.mean() * 1e3 / (nw * N_ST)))
    r = rows[-1]
    print(f"    H={H!s:<4} {stage:10s} {nw:3d} win  {r['mean_ms']:8.3f} +- "
          f"{r['sd_ms']:6.3f} ms   {r['per_station_hour_us']:8.1f} us/station-hour")


print("\n  per horizon and stage")
for H in BENCH_H_LIST:
    g, h = MODELS[H]
    p = PARAMS[H]
    for nw in BENCH_WINDOWS:
        Xw = TEST[H][:nw]
        Z = make_graph_features_dynamic(Xw, graph_ctx, tau=p["tau"], alpha=p["alpha"],
                                        eps=p["eps"], gate_mode="fixed")
        Z2 = Z.reshape(-1, Z.shape[-1])
        D = xgb.DMatrix(np.asarray(flatten_window_per_node(Xw), dtype=np.float32))
        _record(H, "graph", nw, _timeit(lambda: make_graph_features_dynamic(
            Xw, graph_ctx, tau=p["tau"], alpha=p["alpha"], eps=p["eps"], gate_mode="fixed")))
        _record(H, "base", nw, _timeit(lambda: g.predict(Z2)))
        _record(H, "resid", nw, _timeit(lambda: h.predict(D)))
        _record(H, "full", nw, _timeit(lambda: predict_tuned(H, Xw)))

print("\n  all five horizons in sequence")
_X1 = {H: TEST[H][:1] for H in BENCH_H_LIST}
_record("all5", "full", 1, _timeit(
    lambda: [predict_tuned(H, _X1[H]) for H in BENCH_H_LIST]))

bench_df = pd.DataFrame(rows)
BENCH_OUT = os.path.join(OUT_DIR, "deployment_benchmark_tuned.csv")
bench_df.to_csv(BENCH_OUT, index=False)

# footprint
print("\n" + "=" * 78)
print("FOOTPRINT")
print("=" * 78)
size_rows = []
for H in BENCH_H_LIST:
    g, h = MODELS[H]
    s = dict(H=H, base_bytes=len(pickle.dumps(g, protocol=pickle.HIGHEST_PROTOCOL)),
             booster_pickle_bytes=len(pickle.dumps(h, protocol=pickle.HIGHEST_PROTOCOL)),
             booster_ubj_bytes=len(h.save_raw("ubj")),
             num_trees=h.num_boosted_rounds())
    s["total_pickle_bytes"] = len(pickle.dumps([g, h], protocol=pickle.HIGHEST_PROTOCOL))
    size_rows.append(s)
    print(f"  H={H:<2d} base {s['base_bytes']/1024:7.1f} KiB  booster "
          f"{s['booster_ubj_bytes']/1024:8.1f} KiB ({s['num_trees']} rounds)  "
          f"total {s['total_pickle_bytes']/2**20:.3f} MiB")
size_df = pd.DataFrame(size_rows)
tot = size_df["total_pickle_bytes"].sum()
print(f"  five-horizon system: {tot:,} bytes = {tot/2**20:.2f} MiB")
size_df.to_csv(os.path.join(OUT_DIR, "deployment_footprint_tuned.csv"), index=False)
json.dump(ENV, open(os.path.join(OUT_DIR, "deployment_environment_tuned.json"), "w"),
          indent=1, default=str)

print(f"\n  saved {BENCH_OUT}")
print(f"  saved {os.path.join(OUT_DIR, 'deployment_footprint_tuned.csv')}")
print(f"  saved {os.path.join(OUT_DIR, 'deployment_environment_tuned.json')}")

DEPLOYMENT BENCHMARK (tuned DGH-XH)
  processor_model          Intel64 Family 6 Model 165 Stepping 3, GenuineIntel
  logical_cores            12
  memory_gb                15.9
  os                       Windows 10 (10.0.26300)
  python                   3.10.19
  numpy                    2.2.6
  pandas                   2.3.3
  sklearn                  1.7.2
  xgboost                  3.2.0
  blas_thread_env          {'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1'}
  xgboost_predict_threads  1
  warmup_passes            20
  timed_repetitions        200
  timer                    time.perf_counter
  batch_unit               forecast cycle = one 24-h window x all stations
  note                     inference only; models loaded from disk, no training, no I/O in timed region

  tuned parameters from huber_processed_leakfree_fixed_old\all_results_main.csv
    H=1  tau=6  alpha=3.454089  eps=0.035959  csv MAE=6.292063  [ok]
    H=3 